# Validacao - cartas de servico osasco

Objetivo: entender o salto de volume de **432 para 1157** em `gold_carta_servicos`.

Execucao: rodar celula a celula dentro do Fabric (PySpark).

> **Lakehouse:** `lh_cidade_inteligente_osasco`
> **Tabelas investigadas:** `gold_carta_servicos`, `gold_carta_servicos_atualizacoes`

> ⚠️ **Kernel obrigatorio:** selecione **Synapse PySpark** (nao Python 3).
> No VS Code com extensao Fabric: painel superior direito → trocar kernel → Synapse PySpark.

In [ ]:
# Garante que spark existe — no Fabric ja esta disponivel;
# fora do Fabric cria sessao local (tabelas Gold nao estarao disponiveis nesse caso).
try:
    spark
    print(f"spark disponivel: {spark.version} — ambiente Fabric OK")
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder \
        .appName("validacao-carta-servicos-osasco") \
        .getOrCreate()
    print("spark criado localmente — tabelas Gold so funcionam se apontadas para Delta local.")

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 80)

# Helper: executa SQL Spark e retorna pandas para exibicao
def sq(sql: str) -> pd.DataFrame:
    return spark.sql(sql).toPandas()

print("Helper carregado — pode executar as celulas abaixo.")

## Celula 1 — Foto geral: total de linhas vs distinct

**Pergunta:** O total de 1157 e linhas ou servicos unicos?

- Se `total_linhas` >> `total_distinct_nome_servico` → ha duplicidade (por unidade, etapa ou status).
- Se forem proximos → o volume reflete entradas distintas e o aumento pode ser real ou mistura de conceitos.

In [ ]:
sq("""
SELECT
    COUNT(*)                        AS total_linhas,
    COUNT(DISTINCT id_do_servico)   AS distinct_id_servico,
    COUNT(DISTINCT nome_do_servico) AS distinct_nome_servico
FROM gold_carta_servicos
""")

## Celula 2 — Distribuicao por status_tramitacao

**Pergunta:** O salto veio de mais itens Finalizados ou de Em atendimento/Pendente?

- Muitos `Em atendimento` / `Pendente` → o CSV do grid cresceu (mais solicitacoes abertas).
- Concentrado em `Finalizado` → o CSV do BD historico cresceu.

In [ ]:
sq("""
SELECT
    status_tramitacao,
    COUNT(*)                        AS total_linhas,
    COUNT(DISTINCT nome_do_servico) AS distinct_nome_servico
FROM gold_carta_servicos
GROUP BY status_tramitacao
ORDER BY total_linhas DESC
""")

## Celula 3 — Servicos com mais de 1 linha (duplicados por id_do_servico)

**Pergunta:** Algum id_do_servico aparece N vezes?

- Se sim: o `drop_duplicates()` que estava comentado no notebook e a causa direta.

In [ ]:
sq("""
SELECT
    id_do_servico,
    COUNT(*) AS qtd_linhas
FROM gold_carta_servicos
WHERE id_do_servico IS NOT NULL
GROUP BY id_do_servico
HAVING COUNT(*) > 1
ORDER BY qtd_linhas DESC
LIMIT 50
""")

## Celula 4 — Servicos com mais de 1 linha (duplicados por nome_do_servico)

**Pergunta:** Mesmo nome de servico aparece com unidades diferentes?

- Esse era o motivo original do `drop_duplicates()` comentado: servicos com N unidades (escolas, CRAS) geram N linhas.

In [ ]:
sq("""
SELECT
    nome_do_servico,
    COUNT(*)                   AS qtd_linhas,
    COUNT(DISTINCT area_responsavel) AS qtd_areas
FROM gold_carta_servicos
WHERE nome_do_servico IS NOT NULL
GROUP BY nome_do_servico
HAVING COUNT(*) > 1
ORDER BY qtd_linhas DESC
LIMIT 50
""")

## Celula 5 — Inspecionar um servico duplicado

Pegue um `nome_do_servico` da celula 4 e veja todas as suas linhas.

**Troque o valor abaixo pelo servico mais duplicado encontrado.**

In [ ]:
# Troque pelo servico mais duplicado da celula 4
SERVICO_ALVO = "<cole aqui o nome_do_servico mais duplicado>"

sq(f"""
SELECT *
FROM gold_carta_servicos
WHERE nome_do_servico = '{SERVICO_ALVO}'
ORDER BY area_responsavel, status_tramitacao
""")

## Celula 6 — Serie temporal: quando o volume subiu?

**Pergunta:** Em qual data o volume disparou?

- Pico concentrado em uma data → novo CSV carregado com granularidade diferente.
- Crescimento gradual → acumulo de solicitacoes em aberto ao longo do tempo.

In [ ]:
sq("""
SELECT
    CAST(data_consolidada AS DATE)  AS dt,
    status_tramitacao,
    COUNT(*)                        AS qtd_linhas
FROM gold_carta_servicos
GROUP BY CAST(data_consolidada AS DATE), status_tramitacao
ORDER BY dt DESC
LIMIT 60
""")

## Celula 7 — Volume de solicitacoes por ano/mes

Visao agregada para identificar em que periodo o volume subiu.

In [ ]:
sq("""
SELECT
    ano_consolidado,
    mes_consolidado,
    status_tramitacao,
    COUNT(*) AS qtd_linhas
FROM gold_carta_servicos
GROUP BY ano_consolidado, mes_consolidado, status_tramitacao
ORDER BY ano_consolidado DESC, mes_consolidado DESC
""")

## Celula 8 — Tabela de atualizacoes (gold_carta_servicos_atualizacoes)

Essa tabela reflete o grid completo — todos os status (nao so em aberto).

**Pergunta:** Quantas linhas tem e quantas sao solicitacoes unicas?

In [ ]:
sq("""
SELECT
    COUNT(*)                         AS total_linhas,
    COUNT(DISTINCT nome_do_servico)  AS distinct_nome_servico,
    COUNT(DISTINCT no_da_solicitacao) AS distinct_solicitacoes
FROM gold_carta_servicos_atualizacoes
""")

## Celula 9 — Breakdown de status em atualizacoes

Confirma se o grid passou a incluir todos os status (Finalizado inclusive),
o que inflaria o total caso o painel aponte para essa tabela.

In [ ]:
sq("""
SELECT
    status_tramitacao,
    COUNT(*) AS qtd_linhas
FROM gold_carta_servicos_atualizacoes
GROUP BY status_tramitacao
ORDER BY qtd_linhas DESC
""")

## Celula 10 — Schema das tabelas (colunas disponiveis)

Confirma se surgiram colunas novas (ex.: unidade, sigla_unidade) que indicariam
que o CSV passou a ter granularidade de unidade e nao de servico.

In [ ]:
print("=== gold_carta_servicos ===")
spark.sql("DESCRIBE gold_carta_servicos").show(truncate=False)

print("\n=== gold_carta_servicos_atualizacoes ===")
spark.sql("DESCRIBE gold_carta_servicos_atualizacoes").show(truncate=False)

## Celula 11 — Verificar colunas de unidade no CSV origem

Se o CSV `bd_entidade_cadastro_carta.csv` tiver colunas de unidade,
cada unidade gera uma linha separada — causa classica da expansao.

> **Nota:** esta celula le direto do CSV raw no Lakehouse.

In [ ]:
import pandas as pd, csv

ARQUIVO_BD = "/lakehouse/default/Files/raw_cadastro_carta/bd_entidade_cadastro_carta.csv"

bd_raw = pd.read_csv(
    ARQUIVO_BD,
    sep=";",
    encoding="utf-8-sig",
    engine="python",
    quotechar='"',
    doublequote=True,
    quoting=csv.QUOTE_MINIMAL,
    on_bad_lines="skip",
    dtype=str,
    nrows=5,  # so header + 5 linhas para inspecao
)

print(f"Colunas ({len(bd_raw.columns)}):")
print(bd_raw.columns.tolist())

## Celula 12 — Contar linhas dos CSVs raw

Compara o volume bruto dos CSVs com o total da tabela Gold.
Se o BD tiver >> 432 linhas, o crescimento esta na fonte.

In [ ]:
import pandas as pd, csv

ARQUIVO_GRID = "/lakehouse/default/Files/raw_cadastro_carta/grid_cadastro_carta.csv"
ARQUIVO_BD   = "/lakehouse/default/Files/raw_cadastro_carta/bd_entidade_cadastro_carta.csv"

def contar_csv(path):
    df = pd.read_csv(
        path, sep=";", encoding="utf-8-sig", engine="python",
        quotechar='"', doublequote=True, quoting=csv.QUOTE_MINIMAL,
        on_bad_lines="skip", dtype=str,
    )
    return len(df), df.columns.tolist()

n_bd, cols_bd = contar_csv(ARQUIVO_BD)
n_grid, cols_grid = contar_csv(ARQUIVO_GRID)

print(f"bd_entidade_cadastro_carta.csv  : {n_bd:>6} linhas | {len(cols_bd)} colunas")
print(f"grid_cadastro_carta.csv         : {n_grid:>6} linhas | {len(cols_grid)} colunas")
print(f"Total esperado em gold (sem dedup): {n_bd + n_grid}")

total_gold = spark.sql("SELECT COUNT(*) AS n FROM gold_carta_servicos").collect()[0]["n"]
print(f"Total real em gold_carta_servicos: {total_gold}")